In [4]:
# -----------------------------
# 05_Prediction.ipynb
# -----------------------------

# 1. Imports
import joblib
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# -----------------------------

In [5]:

# 2. Load Model
# -----------------------------
# Make sure your trained pipeline is saved at this path
model_path = "../models/spam_pipeline.joblib"
pipeline = joblib.load(model_path)

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [6]:
# -----------------------------
# 3. Preprocessing Function
# -----------------------------
def clean_text(text: str) -> str:
    """
    Clean text by:
    - Lowercasing
    - Removing non-alphanumeric characters
    - Removing extra spaces
    """
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [7]:


# -----------------------------
# 4. Prediction Function
# -----------------------------
def predict_spam(text: str, pipeline=pipeline, top_n=5):
    """
    Predict if the message is Spam or Ham and show top contributing words.
    
    Returns:
        prediction (str): 'Spam' or 'Not Spam'
        top_words (list of tuples): [(word, coefficient), ...]
    """
    # Preprocess
    clean = clean_text(text)
    
    # Predict
    pred = pipeline.predict([clean])[0]
    pred_label = "Spam" if pred == 1 else "Not Spam"
    
    # Get top contributing words
    if hasattr(pipeline.named_steps['clf'], 'coef_'):
        # Get TF-IDF vectorizer
        tfidf = pipeline.named_steps['tfidf']
        clf = pipeline.named_steps['clf']
        
        # Transform text
        X_vect = tfidf.transform([clean])
        
        # Get feature names
        feature_names = np.array(tfidf.get_feature_names_out())
        coefs = clf.coef_[0]  # assuming binary classification
        
        # Compute contribution
        contrib = X_vect.toarray()[0] * coefs
        top_idx = np.argsort(np.abs(contrib))[-top_n:][::-1]
        top_words = [(feature_names[i], contrib[i]) for i in top_idx]
    else:
        top_words = []
    
    return pred_label, top_words

In [8]:

# -----------------------------
# 5. Example Predictions
# -----------------------------
messages = [
    "Congratulations! Claim your free voucher now",
    "Hey, can you call me later?",
    "You have WON 150p, text STOP to unsubscribe",
    "Let's meet for coffee at 5"
]

for msg in messages:
    pred, top_words = predict_spam(msg)
    print(f"Text: {msg}")
    print(f"Prediction: {pred}")
    print(f"Top contributing words: {top_words}")
    print("="*50)


Text: Congratulations! Claim your free voucher now
Prediction: Spam
Top contributing words: [('claim', np.float64(2.697271312110154)), ('voucher', np.float64(1.4542150496025383)), ('free', np.float64(1.2065793216338463)), ('congratulations', np.float64(0.5275898656446313)), ('0808', np.float64(0.0))]
Text: Hey, can you call me later?
Prediction: Not Spam
Top contributing words: [('later', np.float64(-2.2340139121230647)), ('hey', np.float64(-2.020493170122467)), ('08452810073', np.float64(0.0)), ('0808', np.float64(0.0)), ('08001950382', np.float64(0.0))]
Text: You have WON 150p, text STOP to unsubscribe
Prediction: Spam
Top contributing words: [('150p', np.float64(3.3801113535811025)), ('stop', np.float64(1.9187595617131719)), ('won', np.float64(1.911951118441357)), ('text', np.float64(1.6205654292969984)), ('unsubscribe', np.float64(1.4707349712756874))]
Text: Let's meet for coffee at 5
Prediction: Not Spam
Top contributing words: [('let', np.float64(-0.7832080138747671)), ('coffee',

In [3]:


# -----------------------------
# 6. Batch Prediction Example
# -----------------------------
# Suppose you want to predict multiple messages from a CSV
# df = pd.read_csv("../data/raw/test_messages.csv")  # column: 'text'
#  df['clean_text'] = df['text'].apply(clean_text)
#  df['prediction'], df['top_words'] = zip(*df['clean_text'].apply(predict_spam))
#df.head()